# Day 4: Agents and tools

Welcome to Day 4 of the AI agents crash course.

So far: **Day 1** ingest → **Day 2** chunk → **Day 3** index/search. Most real effort is **data prep**; without good data, agents look smart but answer wrong.

Today:

- What makes a system **agentic** (**tool use**)  
- **OpenAI function calling** (Responses API) with a `search` tool  
- **Pydantic AI** to hide the tool loop  

You need **`OPENAI_API_KEY`** (e.g. in `ai_hero/.env`).

## Setup

```bash
uv add openai python-dotenv pydantic-ai minsearch requests python-frontmatter
uv add --dev jupyter
```

This notebook rebuilds a small **FAQ text index** (same idea as Day 3). Extend to vector/hybrid search by swapping what `text_search` calls.

In [1]:
import os
from pathlib import Path

from dotenv import load_dotenv

_p = Path.cwd()
for _ in range(10):
    if (_p / ".env").is_file():
        load_dotenv(_p / ".env")
        break
    if _p.parent == _p:
        break
    _p = _p.parent

if not os.environ.get("OPENAI_API_KEY"):
    raise RuntimeError("Set OPENAI_API_KEY (e.g. in ai_hero/.env).")

from openai import OpenAI

openai_client = OpenAI()

In [2]:
import io
import zipfile

import frontmatter
import requests
from minsearch import Index


def read_repo_data(repo_owner, repo_name):
    url = f"https://codeload.github.com/{repo_owner}/{repo_name}/zip/refs/heads/main"
    resp = requests.get(url)
    if resp.status_code != 200:
        raise RuntimeError(f"Failed to download repository: {resp.status_code}")
    out = []
    zf = zipfile.ZipFile(io.BytesIO(resp.content))
    for file_info in zf.infolist():
        fn = file_info.filename
        fnl = fn.lower()
        if not (fnl.endswith(".md") or fnl.endswith(".mdx")):
            continue
        try:
            with zf.open(file_info) as f_in:
                content = f_in.read().decode("utf-8", errors="ignore")
                post = frontmatter.loads(content)
                data = post.to_dict()
                data["filename"] = fn
                out.append(data)
        except Exception as e:
            print(f"skip {fn}: {e}")
    zf.close()
    return out


dtc_faq = read_repo_data("DataTalksClub", "faq")
de_dtc_faq = [
    d
    for d in dtc_faq
    if "data-engineering" in (d.get("filename") or "").lower()
]

faq_index = Index(text_fields=["question", "content"], keyword_fields=[])
faq_index.fit(de_dtc_faq)


def text_search(query: str):
    """Plain OpenAI section — no type hints required here beyond str."""
    return faq_index.search(query, num_results=5)


len(de_dtc_faq)

498

## 1. Tools and agents

**Agent (working definition):** an LLM that can call **tools** — external functions (here: `search(query)` over your FAQ index).

Without tools, the model only **generates** text from priors — often **generic** for your site-specific policies.

In [3]:
user_prompt = "I just discovered the course, can I join now?"

r_no_tools = openai_client.responses.create(
    model="gpt-4o-mini",
    instructions="You are a helpful assistant.",
    input=user_prompt,
)

print(r_no_tools.output_text)

It depends on the specific course you're interested in. Many courses have specific enrollment periods or prerequisites. If you provide more details about the course, such as the institution or platform, I can help you find the information you need regarding enrollment.


With a **search** tool, the model can pull **real** FAQ rows and answer precisely (see below).

## 2. Function calling with OpenAI (Responses API)

Describe the tool so the model knows **name**, **description**, and **JSON Schema** for arguments.

Flow: **first** `responses.create` → model may emit a **`function_call`** item → you run Python `text_search` → **second** `responses.create` with `previous_response_id` and a **`function_call_output`** item so the model can answer.

In [4]:
import json

text_search_tool = {
    "type": "function",
    "name": "text_search",
    "description": "Search the FAQ database",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ.",
            }
        },
        "required": ["query"],
        "additionalProperties": False,
    },
    "strict": True,
}

In [5]:
system_prompt = """
You are a helpful assistant for a course.
""".strip()

question = "I just discovered the course, can I join now?"

r1 = openai_client.responses.create(
    model="gpt-4o-mini",
    instructions=system_prompt,
    input=question,
    tools=[text_search_tool],
)

[item for item in r1.output if getattr(item, "type", None) == "function_call"]

[ResponseFunctionToolCall(arguments='{"query":"join the course"}', call_id='call_UJtkc5IDzw4ciHJqWFidnQgh', name='text_search', type='function_call', id='fc_053afdc62b9102960069c8a3f53cd88193866a50cc85ff14a4', namespace=None, status='completed')]

In [6]:
call = next(x for x in r1.output if getattr(x, "type", None) == "function_call")
arguments = json.loads(call.arguments)
result = text_search(**arguments)

out_payload = json.dumps(result, default=str)

r2 = openai_client.responses.create(
    model="gpt-4o-mini",
    previous_response_id=r1.id,
    tools=[text_search_tool],
    input=[
        {
            "type": "function_call_output",
            "call_id": call.call_id,
            "output": out_payload,
        }
    ],
)

print(r2.output_text)

Yes, you can still join the course even after the start date! While you may not need to officially register to participate, you will have the opportunity to submit homework. Just keep in mind that there will be deadlines for assignments and the final project, so it's best to stay on top of your work.

For future reference, the next cohort starts on **January 12th, 2026**. You can register before that date using [this link](https://airtable.com/shr6oVXeQvSI5HuWD).


The model is **stateless** per call; **`previous_response_id`** ties the tool result to the prior turn (alternatively, you can resend full structured history — see [function calling](https://platform.openai.com/docs/guides/function-calling)).

## 3. System prompt

**Instructions** steer behavior: when to search, how to handle empty hits, whether to try alternate queries.

Example (stricter retrieval):

```text
You are a helpful assistant for a course.

Use the search tool to find relevant information from the course materials before answering questions.

If you can find specific information through search, use it to provide accurate answers.
If the search doesn't return relevant results, let the user know and provide general guidance.
```

Example (multi-query):

```text
Always search for relevant information before answering.
If the first search doesn't give you enough information, try different search terms.
Make multiple searches if needed to provide comprehensive answers.
```

## 4. Pydantic AI

Frameworks handle **tool loops**, **history**, and **schemas** for you. Here we use **Pydantic AI** ([docs](https://ai.pydantic.dev/)).

Current API (check your version): pass the model as **`openai:gpt-4o-mini`**, register tools, then **`run_sync`** in notebooks (or **`await agent.run(...)`** in async code).

In [7]:
from typing import Any, List

from pydantic_ai import Agent


def text_search_pa(query: str) -> List[Any]:
    """
    Perform a text-based search on the FAQ index.

    Args:
        query: The search query string.

    Returns:
        Up to 5 search results from the FAQ index.
    """
    return faq_index.search(query, num_results=5)


system_prompt_pa = """
You are a helpful assistant for a course.

Use the search tool to find relevant information from the course materials before answering questions.

If you can find specific information through search, use it to provide accurate answers.
If the search doesn't return relevant results, let the user know and provide general guidance.
""".strip()

agent = Agent(
    "openai:gpt-4o-mini",
    name="faq_agent",
    instructions=system_prompt_pa,
    tools=[text_search_pa],
)

question = "I just discovered the course, can I join now?"
result = agent.run_sync(question)

print(result.output)

RuntimeError: This event loop is already running

In [ ]:
result.new_messages()

Inspect **`new_messages()`** / **`all_messages()`** for tool calls and final text.

## Next (Day 5)

**Evaluation** — how good is the agent, prompts, and text vs vector search?

## Homework

- For **your** docs pipeline, expose **`search`** as a tool and chat with the agent.  
- Iterate **system instructions**.  
- Post progress.

[Course](https://alexeygrigorev.com/aihero/)

## Learning in public (examples)

LinkedIn / X: Day 4 — tools + function calling + Pydantic AI; repo link; tomorrow evals.

## Community

[DataTalks.Club Slack](https://datatalks.club/) — `#course-ai-hero`